# Notebook 04D — Project-Integrated Alphalens-Reloaded Diagnostics

## Goal

This notebook evaluates the same TF-IDF and embedding alpha signals using
**Alphalens-Reloaded**.

This notebook is a comparison layer.

The custom notebook remains the baseline:

```text
04C custom diagnostics:
    transparent pandas/numpy implementation

04D Alphalens diagnostics:
    standard factor-analysis tear sheets and Alphalens performance functions
```

This notebook connects to the existing project files:

```text
src/config.py
src/data_ingest.py
src/signal_methods.py
src/alpha_model.py
src/signal_diagnostics_alphalens.py
```

It does **not** replace `run_project.py`.

It stops before optimizer/backtest.

## Cell 1 — Install Alphalens-Reloaded

Run this once in your project environment.

The package name is:

```text
alphalens-reloaded
```

The import name is:

```python
import alphalens
```

In [ ]:

# Run this only if alphalens is not installed.
#
# %pip install -U alphalens-reloaded

## Cell 2 — Imports

Run this notebook from the repo root in VSCode.

The repo root should contain:

```text
run_project.py
src/
data/
outputs/
```

In [ ]:

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Existing project config.
from src.config import (
    PATH_IN,
    SHEET,
    OUT_DIR,
    THEME_TEXT,
    START_DATE,
    END_DATE,
)

# Existing project data functions.
from src.data_ingest import (
    load_universe,
    add_company_descriptions,
    download_prices,
)

# Existing project signal methods.
from src.signal_methods import (
    add_tfidf_similarity,
    add_embeddings,
    add_embedding_similarity,
)

# Existing project alpha construction.
from src.alpha_model import build_alpha

# Project-integrated Alphalens adapter.
from src.signal_diagnostics_alphalens import (
    ALPHALENS_AVAILABLE,
    build_sector_map_from_universe,
    build_alphalens_factor_data_from_prices,
    compute_forward_returns,
    compute_monthly_ic,
    compute_quantile_returns,
    compute_top_bottom_spread,
    compute_sector_exposure,
    alphalens_information_coefficient,
    alphalens_mean_information_coefficient,
    alphalens_mean_return_by_quantile,
)

# Alphalens plotting / tear-sheet modules.
try:
    import alphalens as al
    from alphalens import tears

except Exception as e:
    al = None
    tears = None
    print("Alphalens is not available.")
    print("Install with: pip install alphalens-reloaded")
    print("Error:", repr(e))

print("ALPHALENS_AVAILABLE:", ALPHALENS_AVAILABLE)

## Cell 3 — Confirm project paths

These values come from `src/config.py`.

The notebook uses your project's actual paths instead of generic `data/processed` paths.

In [ ]:

print("PATH_IN:", PATH_IN)
print("SHEET:", SHEET)
print("OUT_DIR:", OUT_DIR)
print("START_DATE:", START_DATE)
print("END_DATE:", END_DATE)

OUT_DIR.mkdir(exist_ok=True)

## Cell 4 — Output paths and runtime switches

The notebook caches slow outputs.

`REBUILD_SCORES = False` means:

```text
load outputs/scored_universe.pkl if it exists
```

`RELOAD_PRICES = False` means:

```text
load outputs/prices.csv if it exists
```

In [ ]:

SCORED_UNIVERSE_PATH = OUT_DIR / "scored_universe.pkl"
PRICES_PATH = OUT_DIR / "prices.csv"

ALPHALENS_DIR = OUT_DIR / "signal_diagnostics_alphalens"
ALPHALENS_DIR.mkdir(exist_ok=True)

REBUILD_SCORES = False
RELOAD_PRICES = False

print("SCORED_UNIVERSE_PATH:", SCORED_UNIVERSE_PATH)
print("PRICES_PATH:", PRICES_PATH)
print("ALPHALENS_DIR:", ALPHALENS_DIR)
print("REBUILD_SCORES:", REBUILD_SCORES)
print("RELOAD_PRICES:", RELOAD_PRICES)

## Cell 5 — Load or build scored universe

This mirrors the early part of `run_project.py`.

If cached scores exist, use them.

Otherwise:

```text
load universe
→ add descriptions
→ build TF-IDF score
→ build embedding score
→ build normalized alphas
```

In [ ]:

if SCORED_UNIVERSE_PATH.exists() and not REBUILD_SCORES:
    # Fast path: load cached scored universe.
    df = pd.read_pickle(SCORED_UNIVERSE_PATH)

    print("Loaded cached scored universe:")
    print(SCORED_UNIVERSE_PATH)

else:
    # Slow path: rebuild scored universe from project inputs.
    print("Building scored universe from scratch.")

    # Load S&P 500 universe from Excel.
    df = load_universe(PATH_IN, SHEET)

    # Add Yahoo Finance company descriptions.
    df = add_company_descriptions(df)

    # Keep rows with usable descriptions.
    df = df[df["Company Description"].astype(str).str.strip() != ""].copy()

    # Compute TF-IDF similarity.
    df = add_tfidf_similarity(df, THEME_TEXT)

    # Build normalized TF-IDF alpha.
    df["alpha_tfidf"] = build_alpha(df["tfidf_similarity"])

    # Compute OpenAI embeddings.
    df = add_embeddings(df)

    # Compute embedding similarity.
    df = add_embedding_similarity(df, THEME_TEXT)

    # Build normalized embedding alpha.
    df["alpha_embedding"] = build_alpha(df["embedding_similarity"])

    # Cache scored universe.
    df.to_pickle(SCORED_UNIVERSE_PATH)

    print("Saved scored universe:")
    print(SCORED_UNIVERSE_PATH)

print("Rows:", len(df))
print("Columns:", list(df.columns))

display(df.head())

## Cell 6 — Build alpha Series

Alphalens ultimately needs a factor indexed by:

```text
date, asset
```

But your project first creates static alpha Series:

```text
ticker -> alpha
```

The adapter repeats static alpha across monthly evaluation dates.

In [ ]:

alpha_tfidf = df.set_index("Ticker")["alpha_tfidf"].dropna()
alpha_embedding = df.set_index("Ticker")["alpha_embedding"].dropna()

alpha_tfidf.index = alpha_tfidf.index.astype(str).str.strip().str.upper()
alpha_embedding.index = alpha_embedding.index.astype(str).str.strip().str.upper()

print("TF-IDF alpha names:", len(alpha_tfidf))
print("Embedding alpha names:", len(alpha_embedding))

display(alpha_tfidf.head())
display(alpha_embedding.head())

## Cell 7 — Load or download prices using project function

This uses the existing project function:

```python
download_prices(...)
```

The notebook does not rewrite Yahoo Finance downloading.

In [ ]:

if PRICES_PATH.exists() and not RELOAD_PRICES:
    # Load cached prices.
    prices = pd.read_csv(PRICES_PATH, index_col=0, parse_dates=True)

    print("Loaded cached prices:")
    print(PRICES_PATH)

else:
    # Download prices for all names used by either signal.
    ticker_list = sorted(set(alpha_tfidf.index).union(set(alpha_embedding.index)))

    prices = download_prices(
        ticker_list,
        START_DATE,
        END_DATE,
    )

    # Cache price matrix.
    prices.to_csv(PRICES_PATH)

    print("Downloaded and saved prices:")
    print(PRICES_PATH)

prices.index = pd.to_datetime(prices.index)
prices = prices.sort_index()
prices.columns = prices.columns.astype(str).str.strip().str.upper()

print("Price matrix shape:", prices.shape)
display(prices.head())

## Cell 8 — Align tickers

Keep only names that exist in:

```text
prices
TF-IDF alpha
embedding alpha
```

In [ ]:

common_tickers = sorted(
    set(prices.columns)
    .intersection(alpha_tfidf.index)
    .intersection(alpha_embedding.index)
)

if len(common_tickers) == 0:
    raise ValueError("No common tickers across prices and alpha signals.")

prices = prices[common_tickers].copy()
alpha_tfidf = alpha_tfidf.loc[common_tickers].copy()
alpha_embedding = alpha_embedding.loc[common_tickers].copy()

print("Common tickers:", len(common_tickers))
print("Prices:", prices.shape)

## Cell 9 — Build sector map

Alphalens can use sectors through the `groupby` argument.

This notebook also saves sector diagnostics directly for comparison.

In [ ]:

try:
    sector_map = build_sector_map_from_universe(df)

    print("Sector map built.")
    print("Number of sectors:", sector_map.nunique())

    display(sector_map.head())

except Exception as e:
    sector_map = None

    print("Could not build sector map.")
    print("Error:", repr(e))

## Cell 10 — Build Alphalens factor_data

This is the main Alphalens step.

It calls:

```python
alphalens.utils.get_clean_factor_and_forward_returns(...)
```

through the project adapter.

In [ ]:

if not ALPHALENS_AVAILABLE:
    raise ImportError(
        "Install Alphalens-Reloaded first: pip install alphalens-reloaded"
    )

PERIODS = (21, 63)

factor_data_embedding = build_alphalens_factor_data_from_prices(
    alpha=alpha_embedding,
    prices=prices,
    periods=PERIODS,
    n_quantiles=5,
    sector_map=sector_map,
    max_loss=0.50,
)

factor_data_tfidf = build_alphalens_factor_data_from_prices(
    alpha=alpha_tfidf,
    prices=prices,
    periods=PERIODS,
    n_quantiles=5,
    sector_map=sector_map,
    max_loss=0.50,
)

print("Embedding factor_data:", factor_data_embedding.shape)
print("TF-IDF factor_data:", factor_data_tfidf.shape)

display(factor_data_embedding.head())

factor_data_embedding.reset_index().to_csv(
    ALPHALENS_DIR / "factor_data_embedding.csv",
    index=False,
)

factor_data_tfidf.reset_index().to_csv(
    ALPHALENS_DIR / "factor_data_tfidf.csv",
    index=False,
)

## Cell 11 — Inspect factor_data

Important columns:

```text
factor
factor_quantile
21D
63D
group, if sector map was available
```

In [ ]:

print("Embedding columns:")
print(list(factor_data_embedding.columns))

print("Embedding quantile counts:")
display(factor_data_embedding["factor_quantile"].value_counts().sort_index())

if "group" in factor_data_embedding.columns:
    print("Embedding group counts:")
    display(factor_data_embedding["group"].value_counts())

print("TF-IDF quantile counts:")
display(factor_data_tfidf["factor_quantile"].value_counts().sort_index())

## Cell 12 — Alphalens full tear sheets

These cells create standard Alphalens tear sheets.

If plotting fails because of package/version issues, continue to the direct performance-function cells below.

In [ ]:

try:
    print("Embedding full tear sheet")
    tears.create_full_tear_sheet(
        factor_data_embedding,
        long_short=True,
        group_neutral=False,
        by_group=("group" in factor_data_embedding.columns),
    )

except Exception as e:
    print("Embedding full tear sheet failed:")
    print(repr(e))

try:
    print("TF-IDF full tear sheet")
    tears.create_full_tear_sheet(
        factor_data_tfidf,
        long_short=True,
        group_neutral=False,
        by_group=("group" in factor_data_tfidf.columns),
    )

except Exception as e:
    print("TF-IDF full tear sheet failed:")
    print(repr(e))

## Cell 13 — Direct Alphalens IC functions

This computes IC tables directly from Alphalens performance functions.

These outputs are easier to save and compare than the tear-sheet plots.

In [ ]:

ic_embedding = alphalens_information_coefficient(factor_data_embedding)
ic_tfidf = alphalens_information_coefficient(factor_data_tfidf)

mean_ic_embedding = alphalens_mean_information_coefficient(
    factor_data_embedding,
    by_time="M",
)

mean_ic_tfidf = alphalens_mean_information_coefficient(
    factor_data_tfidf,
    by_time="M",
)

print("Embedding IC:")
display(ic_embedding.head())

print("Embedding monthly mean IC:")
display(mean_ic_embedding.head())

print("TF-IDF IC:")
display(ic_tfidf.head())

print("TF-IDF monthly mean IC:")
display(mean_ic_tfidf.head())

ic_embedding.to_csv(ALPHALENS_DIR / "alphalens_ic_embedding.csv")
ic_tfidf.to_csv(ALPHALENS_DIR / "alphalens_ic_tfidf.csv")
mean_ic_embedding.to_csv(ALPHALENS_DIR / "alphalens_monthly_mean_ic_embedding.csv")
mean_ic_tfidf.to_csv(ALPHALENS_DIR / "alphalens_monthly_mean_ic_tfidf.csv")

## Cell 14 — Direct Alphalens quantile-return functions

This computes mean return by factor quantile.

Compare this with the custom notebook's quantile-return output.

In [ ]:

mean_qret_embedding, se_qret_embedding = alphalens_mean_return_by_quantile(
    factor_data_embedding,
    by_date=True,
)

mean_qret_tfidf, se_qret_tfidf = alphalens_mean_return_by_quantile(
    factor_data_tfidf,
    by_date=True,
)

print("Embedding mean return by quantile:")
display(mean_qret_embedding.head())

print("TF-IDF mean return by quantile:")
display(mean_qret_tfidf.head())

mean_qret_embedding.to_csv(ALPHALENS_DIR / "alphalens_quantile_returns_embedding.csv")
mean_qret_tfidf.to_csv(ALPHALENS_DIR / "alphalens_quantile_returns_tfidf.csv")

## Cell 15 — Top-minus-bottom spread from Alphalens quantile returns

This converts Alphalens quantile returns into the same idea used in the custom notebook:

```text
top quantile return - bottom quantile return
```

In [ ]:

def spread_from_alphalens_mean_quantile_returns(mean_qret):
    """Compute top-minus-bottom spread from Alphalens mean quantile returns."""

    if not isinstance(mean_qret.index, pd.MultiIndex):
        raise ValueError("Expected MultiIndex from Alphalens mean_return_by_quantile.")

    quantile_level = mean_qret.index.names[1]

    spreads = {}

    for column in mean_qret.columns:
        wide = mean_qret[column].unstack(quantile_level)

        bottom_quantile = wide.columns.min()
        top_quantile = wide.columns.max()

        spreads[column] = wide[top_quantile] - wide[bottom_quantile]

    spread = pd.DataFrame(spreads)
    spread.index.name = "date"

    return spread


spread_embedding = spread_from_alphalens_mean_quantile_returns(mean_qret_embedding)
spread_tfidf = spread_from_alphalens_mean_quantile_returns(mean_qret_tfidf)

print("Embedding Alphalens spread:")
display(spread_embedding.head())

print("TF-IDF Alphalens spread:")
display(spread_tfidf.head())

spread_embedding.to_csv(ALPHALENS_DIR / "alphalens_top_bottom_spread_embedding.csv")
spread_tfidf.to_csv(ALPHALENS_DIR / "alphalens_top_bottom_spread_tfidf.csv")

## Cell 16 — Plot Alphalens monthly IC and spread

These plots give a quick visual comparison.

In [ ]:

ax = mean_ic_embedding.plot(
    figsize=(10, 5),
    title="Alphalens Monthly Mean IC — Embedding Alpha",
)

ax.axhline(0.0, linestyle="--", linewidth=1)
ax.set_xlabel("Date")
ax.set_ylabel("Mean IC")
plt.tight_layout()
plt.show()

ax = mean_ic_tfidf.plot(
    figsize=(10, 5),
    title="Alphalens Monthly Mean IC — TF-IDF Alpha",
)

ax.axhline(0.0, linestyle="--", linewidth=1)
ax.set_xlabel("Date")
ax.set_ylabel("Mean IC")
plt.tight_layout()
plt.show()

ax = spread_embedding.plot(
    figsize=(10, 5),
    title="Alphalens Top-minus-Bottom Spread — Embedding Alpha",
)

ax.axhline(0.0, linestyle="--", linewidth=1)
ax.set_xlabel("Date")
ax.set_ylabel("Forward return spread")
plt.tight_layout()
plt.show()

ax = spread_tfidf.plot(
    figsize=(10, 5),
    title="Alphalens Top-minus-Bottom Spread — TF-IDF Alpha",
)

ax.axhline(0.0, linestyle="--", linewidth=1)
ax.set_xlabel("Date")
ax.set_ylabel("Forward return spread")
plt.tight_layout()
plt.show()

## Cell 17 — Same public function signatures as custom baseline

The adapter also exposes the same function names as the custom baseline.

These are useful if you want to compare custom vs Alphalens style with the same calls.

In [ ]:

forward_returns = compute_forward_returns(
    prices=prices,
    horizons=PERIODS,
)

monthly_ic_embedding = compute_monthly_ic(
    alpha=alpha_embedding,
    forward_returns=forward_returns,
    method="spearman",
)

monthly_ic_tfidf = compute_monthly_ic(
    alpha=alpha_tfidf,
    forward_returns=forward_returns,
    method="spearman",
)

quantile_returns_embedding = compute_quantile_returns(
    alpha=alpha_embedding,
    forward_returns=forward_returns,
    n_quantiles=5,
)

quantile_returns_tfidf = compute_quantile_returns(
    alpha=alpha_tfidf,
    forward_returns=forward_returns,
    n_quantiles=5,
)

spread_function_embedding = compute_top_bottom_spread(
    quantile_returns_embedding
)

spread_function_tfidf = compute_top_bottom_spread(
    quantile_returns_tfidf
)

display(monthly_ic_embedding.head())
display(quantile_returns_embedding.head())
display(spread_function_embedding.head())

## Cell 18 — Sector exposure comparison

Alphalens uses sector information through factor_data `group`.

This standalone sector summary mirrors the custom baseline.

In [ ]:

if sector_map is not None:
    sector_exposure_embedding = compute_sector_exposure(
        alpha_embedding,
        sector_map,
    )

    sector_exposure_tfidf = compute_sector_exposure(
        alpha_tfidf,
        sector_map,
    )

    print("Embedding sector exposure:")
    display(sector_exposure_embedding)

    print("TF-IDF sector exposure:")
    display(sector_exposure_tfidf)

    sector_exposure_embedding.to_csv(
        ALPHALENS_DIR / "sector_exposure_embedding.csv"
    )

    sector_exposure_tfidf.to_csv(
        ALPHALENS_DIR / "sector_exposure_tfidf.csv"
    )

else:
    print("No sector map available.")

## Cell 19 — Final comparison summary

This compact table is useful for comparing against the custom diagnostics notebook.

In [ ]:

summary = pd.DataFrame(
    {
        "alphalens_embedding_mean_ic": mean_ic_embedding.mean(),
        "alphalens_tfidf_mean_ic": mean_ic_tfidf.mean(),
        "alphalens_embedding_mean_spread": spread_embedding.mean(),
        "alphalens_tfidf_mean_spread": spread_tfidf.mean(),
    }
)

display(summary)

summary.to_csv(ALPHALENS_DIR / "alphalens_diagnostic_summary.csv")

## Cell 20 — Interpretation

Use this notebook as the standard-package benchmark.

Compare against the custom baseline notebook:

```text
custom monthly IC
vs.
Alphalens monthly mean IC

custom quantile returns
vs.
Alphalens mean_return_by_quantile

custom top-minus-bottom spread
vs.
Alphalens spread from quantile returns

custom sector exposure
vs.
Alphalens group analysis / sector exposure
```

If results differ, check:

```text
date alignment
forward return horizon definitions
quantile assignment
demeaning / group-neutral settings
dropped rows from Alphalens max_loss
```